# TP3 : Ingénierie Deep Learning — TensorFlow vs PyTorch
**D1 — Session 2026**

Chaque cellule est indépendante. Lancez-les une par une avec **Shift+Enter**.

## Section 1 — Configuration de l'Environnement

In [ ]:
import torch
import tensorflow as tf
import numpy as np

print(f"PyTorch Version    : {torch.__version__} | CUDA Available : {torch.cuda.is_available()}")
print(f"MPS Available      : {torch.backends.mps.is_available()}  (Apple Silicon)")
print(f"TensorFlow Version : {tf.__version__} | GPU Detected : {tf.config.list_physical_devices('GPU')}")

# Détection device optimal
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f"\nDevice PyTorch sélectionné : {DEVICE}")

---
## Partie 1 — Fondations et Algèbre Tensorielle
### Exercice 1 : Initialisation comparée

In [ ]:
# ── 1.1 Tenseur de zéros 3×4 ──
zeros_pt = torch.zeros(3, 4)
zeros_tf = tf.zeros((3, 4))
print('Zéros PyTorch :\n', zeros_pt)
print('Zéros TensorFlow :\n', zeros_tf.numpy())

In [ ]:
# ── 1.2 Matrice identité 5×5 ──
eye_pt = torch.eye(5)
eye_tf = tf.eye(5)
print('Identité PyTorch :\n', eye_pt)
print('Identité TensorFlow :\n', eye_tf.numpy())

In [ ]:
# ── 1.3 Tenseur aléatoire N(0,1) de taille 2×3×3 ──
torch.manual_seed(42)
tf.random.set_seed(42)

rand_pt = torch.randn(2, 3, 3)
rand_tf = tf.random.normal((2, 3, 3))

print('Aléatoire PyTorch (seed=42) :\n', rand_pt)
print('Aléatoire TensorFlow (seed=42) :\n', rand_tf.numpy())

In [ ]:
# ── 1.4 Reproductibilité : vérification seed fixe ──
torch.manual_seed(42)
a1 = torch.randn(3)
torch.manual_seed(42)
a2 = torch.randn(3)
print('Reproductibilité PyTorch (seed=42) :', torch.allclose(a1, a2))  # True

tf.random.set_seed(42)
b1 = tf.random.normal((3,))
tf.random.set_seed(42)
b2 = tf.random.normal((3,))
print('Reproductibilité TensorFlow (seed=42) :', np.allclose(b1.numpy(), b2.numpy()))  # True

### Exercice 2 : Typage, Interopérabilité NumPy et Reshaping

In [ ]:
# ── 2.1 Conversion NumPy → tenseur ──
arr = np.arange(1, 13).reshape(3, 4)
print('NumPy original :\n', arr)

tensor_pt = torch.tensor(arr, dtype=torch.float32)
tensor_tf = tf.constant(arr, dtype=tf.float32)

print('\nPyTorch float32 :\n', tensor_pt)
print('dtype :', tensor_pt.dtype)
print('\nTensorFlow float32 :\n', tensor_tf.numpy())
print('dtype :', tensor_tf.dtype)

In [ ]:
# ── 2.2 Partage mémoire vs copie ──
arr_shared = np.arange(1, 13, dtype=np.float32).reshape(3, 4)

# PyTorch : from_numpy() PARTAGE la mémoire
pt_shared = torch.from_numpy(arr_shared)
arr_shared[0, 0] = 999.0
print('PyTorch from_numpy — partage mémoire :')
print('  arr modifié [0,0] =', arr_shared[0, 0])
print('  tensor[0,0] =', pt_shared[0, 0].item(), '← modifié aussi ✓')

# torch.tensor() COPIE
arr2 = np.arange(1, 13, dtype=np.float32).reshape(3, 4)
pt_copy = torch.tensor(arr2)
arr2[0, 0] = 999.0
print('\nPyTorch tensor() — copie :')
print('  arr modifié [0,0] =', arr2[0, 0])
print('  tensor[0,0] =', pt_copy[0, 0].item(), '← inchangé ✓')

# TensorFlow : toujours une copie
arr3 = np.arange(1, 13, dtype=np.float32).reshape(3, 4)
tf_const = tf.constant(arr3)
arr3[0, 0] = 999.0
print('\nTensorFlow tf.constant() — toujours copie :')
print('  arr modifié [0,0] =', arr3[0, 0])
print('  tensor[0,0] =', tf_const[0, 0].numpy(), '← inchangé ✓')

In [ ]:
# ── 2.3 Reshape 3×4 → 2×6, puis flatten 1D ──
arr = np.arange(1, 13).reshape(3, 4).astype(np.float32)
pt  = torch.tensor(arr)
tft = tf.constant(arr)

# Reshape
pt_2x6  = pt.view(2, 6)          # PyTorch : view (vue, mémoire partagée)
tf_2x6  = tf.reshape(tft, (2, 6))  # TensorFlow
print('Reshape 2×6 PyTorch :\n', pt_2x6)
print('Reshape 2×6 TensorFlow :\n', tf_2x6.numpy())

# Flatten
pt_flat  = pt.reshape(-1)          # ou pt.flatten()
tf_flat  = tf.reshape(tft, [-1])   # ou tf.keras.layers.Flatten()
print('\nFlatten PyTorch :', pt_flat)
print('Flatten TensorFlow :', tf_flat.numpy())

---
## Partie 2 — Graphes de Calcul et Différenciation Automatique
### Exercice 3 : Optimisation de Rosenbrock par descente de gradient
**f(x,y) = (a-x)² + b(y-x²)²** avec a=1, b=100, minimum en (1,1)

In [ ]:
# ── 3.1 Mission PyTorch — Autograd ──
import matplotlib.pyplot as plt

a, b = 1.0, 100.0
eta  = 0.002
n_iter = 100

x_pt = torch.tensor(0.0, requires_grad=True)
y_pt = torch.tensor(0.0, requires_grad=True)

history_pt = []
for i in range(n_iter):
    f = (a - x_pt)**2 + b * (y_pt - x_pt**2)**2
    f.backward()                          # calcule df/dx et df/dy

    with torch.no_grad():
        x_pt -= eta * x_pt.grad
        y_pt -= eta * y_pt.grad

    x_pt.grad.zero_()
    y_pt.grad.zero_()

    history_pt.append(f.item())

print(f'PyTorch — Résultat après {n_iter} itérations :')
print(f'  x = {x_pt.item():.6f} (cible : 1.0)')
print(f'  y = {y_pt.item():.6f} (cible : 1.0)')
print(f'  f(x,y) = {history_pt[-1]:.6e}')

plt.figure(figsize=(8, 4))
plt.semilogy(history_pt, label='PyTorch')
plt.xlabel('Itération') ; plt.ylabel('f(x,y) [log]')
plt.title('Convergence Rosenbrock — PyTorch') ; plt.legend() ; plt.grid(True)
plt.tight_layout() ; plt.show()

In [ ]:
# ── 3.2 Mission TensorFlow — GradientTape ──
a, b = 1.0, 100.0
eta  = 0.002

x_tf = tf.Variable(0.0, dtype=tf.float32)
y_tf = tf.Variable(0.0, dtype=tf.float32)

history_tf = []
for i in range(100):
    with tf.GradientTape() as tape:
        f = (a - x_tf)**2 + b * (y_tf - x_tf**2)**2

    grads = tape.gradient(f, [x_tf, y_tf])  # [df/dx, df/dy]
    x_tf.assign_sub(eta * grads[0])
    y_tf.assign_sub(eta * grads[1])

    history_tf.append(f.numpy())

print('TensorFlow — Résultat après 100 itérations :')
print(f'  x = {x_tf.numpy():.6f} (cible : 1.0)')
print(f'  y = {y_tf.numpy():.6f} (cible : 1.0)')
print(f'  f(x,y) = {history_tf[-1]:.6e}')

plt.figure(figsize=(8, 4))
plt.semilogy(history_pt, label='PyTorch', linestyle='--')
plt.semilogy(history_tf, label='TensorFlow')
plt.xlabel('Itération') ; plt.ylabel('f(x,y) [log]')
plt.title('Convergence Rosenbrock — Comparaison') ; plt.legend() ; plt.grid(True)
plt.tight_layout() ; plt.show()

---
## Partie 3 — Pipeline de Classification d'Images
### Exercice 4 : Architecture CNN (MNIST from scratch)

In [ ]:
# ── 4.1 Chargement MNIST ──
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f'Train : {len(train_ds)} images | Test : {len(test_ds)} images')
print(f'Shape d\'un batch : {next(iter(train_loader))[0].shape}')

In [ ]:
# ── 4.2 Architecture CNN PyTorch ──
class CNNPyTorch(nn.Module):
    """
    Conv(32, 3x3, ReLU) → MaxPool(2x2) → Flatten → FC(128, ReLU) → FC(10, Logits)
    """
    def __init__(self) -> None:
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 14 * 14, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.conv(x))

model_pt = CNNPyTorch().to(DEVICE)
print(model_pt)
n_params = sum(p.numel() for p in model_pt.parameters())
print(f'\nNombre de paramètres : {n_params:,}')

In [ ]:
# ── 4.3 Architecture CNN TensorFlow ──
class CNNTensorFlow(tf.keras.Model):  # type: ignore
    """
    Conv(32, 3x3, ReLU) → MaxPool(2x2) → Flatten → FC(128, ReLU) → FC(10, Logits)
    """
    def __init__(self) -> None:
        super().__init__()
        self.conv1   = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')
        self.pool    = tf.keras.layers.MaxPooling2D((2, 2))
        self.flatten = tf.keras.layers.Flatten()
        self.fc1     = tf.keras.layers.Dense(128, activation='relu')
        self.fc2     = tf.keras.layers.Dense(10)   # logits

    def call(self, x: tf.Tensor, training: bool = False) -> tf.Tensor:  # type: ignore
        x = self.pool(self.conv1(x))
        return self.fc2(self.fc1(self.flatten(x)))

model_tf = CNNTensorFlow()
# Build avec un batch factice pour afficher le résumé
model_tf(tf.zeros((1, 28, 28, 1)))
model_tf.summary()

### Exercice 5 : Boucle d'Entraînement Personnalisée

In [ ]:
# ── 5.1 Boucle d'entraînement PyTorch (1 époque) ──
criterion = nn.CrossEntropyLoss()
optimizer_pt = torch.optim.Adam(model_pt.parameters(), lr=1e-3)

def train_epoch_pytorch(model: CNNPyTorch, loader: DataLoader,  # type: ignore
                        criterion: nn.Module, optimizer: torch.optim.Optimizer) -> tuple[float, float]:
    model.train()                     # mode entraînement (active dropout/BN)
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # 1. Forward pass
        logits = model(images)

        # 2. Calcul de la perte
        loss = criterion(logits, labels)

        # 3. Remise à zéro des gradients
        optimizer.zero_grad()

        # 4. Rétropropagation
        loss.backward()

        # 5. Mise à jour des poids (Adam)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, 100 * correct / total

print('Entraînement PyTorch (1 époque)...')
loss_pt, acc_pt = train_epoch_pytorch(model_pt, train_loader, criterion, optimizer_pt)
print(f'  Loss = {loss_pt:.4f} | Accuracy = {acc_pt:.2f}%')

In [ ]:
# ── 5.2 Boucle d'entraînement TensorFlow (1 époque) ──
# Chargement MNIST via TF (format NHWC : batch, height, width, channels)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[..., tf.newaxis].astype('float32') / 255.0
x_test  = x_test[..., tf.newaxis].astype('float32') / 255.0

train_tf = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(64)

loss_fn_tf   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer_tf = tf.keras.optimizers.Adam(learning_rate=1e-3)

def train_epoch_tensorflow(
    model: CNNTensorFlow,
    dataset: tf.data.Dataset,
    loss_fn: tf.keras.losses.Loss,
    optimizer: tf.keras.optimizers.Optimizer,
) -> tuple[float, float]:
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in dataset:
        with tf.GradientTape() as tape:
            # 1. Forward pass
            logits = model(images, training=True)
            # 2. Calcul de la perte
            loss = loss_fn(labels, logits)

        # 3+4. Gradients et rétropropagation
        grads = tape.gradient(loss, model.trainable_variables)

        # 5. Mise à jour Adam
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        preds   = tf.argmax(logits, axis=1)
        correct += tf.reduce_sum(tf.cast(preds == labels, tf.int32)).numpy()
        total   += images.shape[0]
        total_loss += loss.numpy() * images.shape[0]

    return total_loss / total, 100 * correct / total

print('Entraînement TensorFlow (1 époque)...')
loss_tff, acc_tff = train_epoch_tensorflow(model_tf, train_tf, loss_fn_tf, optimizer_tf)
print(f'  Loss = {loss_tff:.4f} | Accuracy = {acc_tff:.2f}%')

In [ ]:
# ── 5.3 Évaluation sur le jeu de test (PyTorch) ──
def evaluate_pytorch(model: CNNPyTorch, loader: DataLoader) -> float:  # type: ignore
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return 100 * correct / total

acc_test_pt = evaluate_pytorch(model_pt, test_loader)
print(f'Test Accuracy PyTorch  : {acc_test_pt:.2f}%')

# TensorFlow
x_test_tf = x_test
logits_tf  = model_tf(x_test_tf, training=False)
preds_tf   = tf.argmax(logits_tf, axis=1)
acc_test_tf = 100 * tf.reduce_mean(tf.cast(preds_tf == y_test, tf.float32)).numpy()
print(f'Test Accuracy TensorFlow : {acc_test_tf:.2f}%')

---
## Partie 4 — Gestion Avancée du Matériel
### Exercice 6 : Migration Multi-Device (CPU ↔ GPU)

In [ ]:
# ── 6.1 PyTorch — Détection et transfert device ──
print(f'Device disponible : {DEVICE}')

# Modèle déjà sur DEVICE — un mini-batch migré aussi
images_batch, labels_batch = next(iter(test_loader))
images_batch = images_batch.to(DEVICE)
labels_batch = labels_batch.to(DEVICE)

model_pt.eval()
with torch.no_grad():
    out = model_pt(images_batch)
print(f'Inférence sur {DEVICE} — logits shape : {out.shape}')

# Conversion vers NumPy : nécessite .cpu() si le tenseur est sur GPU/MPS
try:
    arr_direct = out.numpy()   # Échoue si device != cpu
    print('Conversion directe réussie (cpu)')
except RuntimeError as e:
    print(f'Erreur attendue ({DEVICE}) : {type(e).__name__}')
    arr_cpu = out.cpu().numpy()
    print(f'Après .cpu().numpy() — shape : {arr_cpu.shape} ✓')

In [ ]:
# ── 6.2 TensorFlow — Placement explicite des opérations ──
gpus = tf.config.list_physical_devices('GPU')
device_name = '/GPU:0' if gpus else '/CPU:0'
print(f'Devices TF disponibles : {[d.name for d in tf.config.list_physical_devices()]}')
print(f'Device forcé : {device_name}')

with tf.device(device_name):
    x_sample = tf.constant(x_test[:10])
    logits = model_tf(x_sample, training=False)
    print(f'Inférence TF sur {device_name} — logits shape : {logits.shape}')

In [ ]:
# ── 6.3 Question de réflexion — GPU tenseur → NumPy ──
print("""
PYTORCH — Pourquoi .cpu() est nécessaire avant .numpy() :
  Un tenseur GPU réside en VRAM (mémoire vidéo). NumPy opère
  exclusivement en RAM (mémoire hôte). Sans .cpu(), PyTorch ne peut
  pas accéder à la VRAM via l'interface NumPy → RuntimeError.
  Solution : tensor.cpu().detach().numpy()

TENSORFLOW — Comportement équivalent :
  tf.Tensor.numpy() fait automatiquement le transfert GPU→CPU
  (copy implicite) sans lever d'erreur. Plus permissif, mais cache
  le coût du transfert mémoire (VRAM → RAM).
""")

---
## Récapitulatif Comparatif PyTorch vs TensorFlow

In [1]:
import pandas as pd

comparatif = {
    'Aspect': [
        'Graphe de calcul',
        'Différentiation',
        'Couche Convolutive',
        'Mode entraînement',
        'Loss classification',
        'Optimiseur Adam',
        'Zéro gradients',
        'Rétropropagation',
        'GPU → NumPy',
        'Format images',
    ],
    'PyTorch': [
        'Dynamique (Eager par défaut)',
        'tensor.backward() + .grad',
        'nn.Conv2d(in_ch, out_ch, k)',
        'model.train()',
        'nn.CrossEntropyLoss()',
        'torch.optim.Adam(params, lr)',
        'optimizer.zero_grad()',
        'loss.backward()',
        'tensor.cpu().numpy()',
        'NCHW (N, C, H, W)',
    ],
    'TensorFlow': [
        'Statique → Eager + @tf.function',
        'tf.GradientTape() + tape.gradient()',
        'tf.keras.layers.Conv2D(filters, k)',
        'model(x, training=True)',
        'SparseCategoricalCrossentropy(from_logits=True)',
        'tf.keras.optimizers.Adam(lr)',
        'Automatique (dans GradientTape)',
        'grads = tape.gradient(loss, vars)',
        'tensor.numpy() (copie implicite)',
        'NHWC (N, H, W, C)',
    ],
}

df = pd.DataFrame(comparatif)
display(df.style.set_properties(**{'text-align': 'left'}))

,Aspect,PyTorch,TensorFlow
0,Graphe de calcul,Dynamique (Eager par défaut),Statique → Eager + @tf.function
1,Différentiation,tensor.backward() + .grad,tf.GradientTape() + tape.gradient()
2,Couche Convolutive,"nn.Conv2d(in_ch, out_ch, k)","tf.keras.layers.Conv2D(filters, k)"
3,Mode entraînement,model.train(),"model(x, training=True)"
4,Loss classification,nn.CrossEntropyLoss(),SparseCategoricalCrossentropy(from_logits=True)
5,Optimiseur Adam,"torch.optim.Adam(params, lr)",tf.keras.optimizers.Adam(lr)
6,Zéro gradients,optimizer.zero_grad(),Automatique (dans GradientTape)
7,Rétropropagation,loss.backward(),"grads = tape.gradient(loss, vars)"
8,GPU → NumPy,tensor.cpu().numpy(),tensor.numpy() (copie implicite)
9,Format images,"NCHW (N, C, H, W)","NHWC (N, H, W, C)"
